In [ ]:
# Prompt Engineering Fundamentals
# ====================================
#
# Systematic comparison of core prompting techniques:
# - Zero-shot: task instruction only, no examples
# - Few-shot: examples teach the pattern
# - Role-based: persona shapes the response
# - Structured output: consistent formats for downstream processing
#
# All techniques demonstrated with P&C insurance scenarios.

import boto3
import json
import time

# ---- Bedrock client ----
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

def ask(prompt, system=None, max_tokens=1024, temperature=0.0):
    """Clean wrapper for Bedrock Converse API."""
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]
    
    response = bedrock.converse(**kwargs)
    
    return {
        'text': response['output']['message']['content'][0]['text'],
        'input_tokens': response['usage']['inputTokens'],
        'output_tokens': response['usage']['outputTokens'],
        'stop_reason': response['stopReason']
    }


# ============================================
# ZERO-SHOT PROMPTING
# ============================================
# No examples — just tell Claude what to do.
# Works well for straightforward tasks where
# the instruction is clear enough on its own.

print("=" * 65)
print("ZERO-SHOT PROMPTING")
print("=" * 65)

# ---- Sample claim for all tasks ----
claim_text = """The insured reports that on February 15, 2026, their 2022 Toyota Camry 
was parked at a shopping mall in Saskatoon when a shopping cart was blown by high winds 
into the driver's side door, causing a large dent and paint damage. No other vehicles 
were involved. Estimated repair cost is $2,800."""

print(f"\nSample Claim:\n{claim_text}\n")

# ---- Task 1: Claim classification ----
print("--- Task 1: Claim Classification ---")
result = ask(f"Classify this insurance claim as one of: Collision, Comprehensive, Liability, or Not Covered.\n\nCLAIM:\n{claim_text}")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

# ---- Task 2: Severity assessment ----
print("\n--- Task 2: Severity Assessment ---")
result = ask(f"Rate the severity of this insurance claim as Low, Medium, or High and explain why in one sentence.\n\nCLAIM:\n{claim_text}")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

# ---- Task 3: Summarization ----
print("\n--- Task 3: Claim Summary ---")
result = ask(f"Summarize this insurance claim in exactly two sentences.\n\nCLAIM:\n{claim_text}")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

# ---- Task 4: Information extraction ----
print("\n--- Task 4: Information Extraction ---")
result = ask(f"Extract the following from this claim: date of loss, vehicle, location, cause of damage, and estimated cost.\n\nCLAIM:\n{claim_text}")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

print(f"\n{'=' * 65}")
print("Zero-shot works well for clear, well-defined tasks.")
print("No examples needed — the instruction does the heavy lifting.")
print(f"{'=' * 65}")

In [ ]:
# ============================================
# FEW-SHOT PROMPTING
# ============================================
# Provide examples so Claude learns the pattern.
# This gives you consistent format AND better
# accuracy on domain-specific tasks.

print("=" * 65)
print("FEW-SHOT PROMPTING")
print("=" * 65)

# ---- Task 1: Claim Classification (with examples) ----
print("\n--- Task 1: Claim Classification (few-shot) ---")

few_shot_classify = """Classify insurance claims using EXACTLY this format:
Classification: [type]
Confidence: [High/Medium/Low]
Reason: [one sentence]

EXAMPLES:

Claim: The insured rear-ended another vehicle at a red light, causing bumper damage to both cars.
Classification: Collision
Confidence: High
Reason: Direct impact between two vehicles while in motion is a collision event.

Claim: The insured's vehicle was stolen from their driveway overnight.
Classification: Comprehensive
Confidence: High
Reason: Theft is a non-collision peril covered under comprehensive coverage.

Claim: A pedestrian tripped on the insured's property and broke their wrist.
Classification: Liability
Confidence: High
Reason: Bodily injury to a third party on the insured's property is a liability claim.

Now classify this claim:

Claim: """ + claim_text

result_fewshot = ask(few_shot_classify)
print(f"Answer:\n{result_fewshot['text']}")
print(f"Tokens: {result_fewshot['input_tokens']} in / {result_fewshot['output_tokens']} out")


# ---- Compare: zero-shot vs few-shot on a tricky claim ----
print(f"\n\n--- Tricky Claim: Zero-Shot vs Few-Shot ---")

tricky_claim = """The insured was driving on Highway 11 near Prince Albert when a deer 
jumped onto the road. The insured swerved to avoid the deer and struck a guardrail. 
The front bumper, hood, and right fender are damaged. No injuries reported. 
Estimated repair cost is $8,500."""

print(f"\nTricky Claim:\n{tricky_claim}\n")

# Zero-shot
print("[ZERO-SHOT]")
result_zero = ask(f"Classify this insurance claim as one of: Collision, Comprehensive, Liability, or Not Covered.\n\nCLAIM:\n{tricky_claim}")
print(f"Answer: {result_zero['text'][:200]}")
print(f"Tokens: {result_zero['input_tokens']} in / {result_zero['output_tokens']} out")

# Few-shot
print("\n[FEW-SHOT]")
few_shot_tricky = """Classify insurance claims using EXACTLY this format:
Classification: [type]
Confidence: [High/Medium/Low]
Reason: [one sentence]

EXAMPLES:

Claim: The insured rear-ended another vehicle at a red light, causing bumper damage to both cars.
Classification: Collision
Confidence: High
Reason: Direct impact between two vehicles while in motion is a collision event.

Claim: The insured's vehicle was stolen from their driveway overnight.
Classification: Comprehensive
Confidence: High
Reason: Theft is a non-collision peril covered under comprehensive coverage.

Claim: A tree branch fell on the insured's car during a thunderstorm.
Classification: Comprehensive
Confidence: High
Reason: Falling objects due to weather are non-collision perils under comprehensive coverage.

Claim: The insured swerved to avoid a dog and hit a telephone pole.
Classification: Collision
Confidence: High
Reason: Impact with a fixed object (telephone pole) is classified as collision regardless of why the driver swerved.

Now classify this claim:

Claim: """ + tricky_claim

result_few = ask(few_shot_tricky)
print(f"Answer:\n{result_few['text']}")
print(f"Tokens: {result_few['input_tokens']} in / {result_few['output_tokens']} out")


# ---- Few-shot for structured extraction ----
print(f"\n\n--- Task 2: Structured Extraction (few-shot) ---")

few_shot_extract = """Extract claim details using EXACTLY this format:

CLAIM: A driver hit a parked car in a grocery store lot on Jan 5, 2026. The parked car had bumper damage. Repair estimate $1,200.
DATE: 2026-01-05
VEHICLE_DAMAGED: Parked car (details not specified)
LOCATION: Grocery store parking lot
PERIL: Collision with parked vehicle
ESTIMATED_COST: $1,200
INJURIES: None reported
THIRD_PARTY: Yes — parked car owner

CLAIM: On March 10, 2026, hail damaged the insured's 2023 Ford Explorer in their driveway. Dents on hood and roof. Repair estimate $4,100.
DATE: 2026-03-10
VEHICLE_DAMAGED: 2023 Ford Explorer
LOCATION: Insured's driveway
PERIL: Hail damage
ESTIMATED_COST: $4,100
INJURIES: None reported
THIRD_PARTY: No

Now extract from this claim:

CLAIM: """ + claim_text

result_extract = ask(few_shot_extract)
print(f"Answer:\n{result_extract['text']}")
print(f"Tokens: {result_extract['input_tokens']} in / {result_extract['output_tokens']} out")


print(f"\n{'=' * 65}")
print("Few-shot gives you two advantages over zero-shot:")
print("  1. Consistent formatting — output matches your examples")
print("  2. Better accuracy on edge cases — examples teach the pattern")
print(f"{'=' * 65}")

In [ ]:
# ============================================
# ROLE-BASED PROMPTING
# ============================================
# The system prompt persona changes HOW Claude responds.
# Same claim, different perspectives — like asking three
# different people in the office the same question.

print("=" * 65)
print("ROLE-BASED PROMPTING")
print("=" * 65)

# ---- Same claim, three personas ----
complex_claim = """The insured's 2024 Ram ProMaster 2500 was involved in an accident on 
January 28, 2026 while making a delivery in downtown Regina. The driver reports that a 
cyclist cut in front of the van at an intersection, causing the driver to brake suddenly 
and rear-end the vehicle ahead. The cyclist left the scene. Damage to the insured's van 
includes front bumper and hood damage ($6,200). The vehicle ahead sustained rear bumper 
damage ($3,800). The other driver is reporting neck pain. A police report was filed 
(RPS-2026-0892). The insured's driver has two prior moving violations in the past 24 months."""

print(f"\nClaim:\n{complex_claim}\n")
print("=" * 65)

# ---- Persona 1: Claims Adjuster ----
print("\n--- Persona 1: Claims Adjuster ---")
adjuster_system = """You are a senior P&C claims adjuster with 15 years of experience. 
Focus on coverage determination, liability assessment, and next steps for the claim. 
Be concise and action-oriented. Use industry terminology."""

result = ask(complex_claim + "\n\nProvide your assessment of this claim.", system=adjuster_system)
print(f"System: Senior Claims Adjuster")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

# ---- Persona 2: Underwriter ----
print("\n\n--- Persona 2: Underwriter ---")
underwriter_system = """You are a commercial auto underwriter. Assess this claim from a 
risk management perspective. Focus on what this claim tells you about the insured's risk 
profile and any implications for renewal pricing. Be concise."""

result = ask(complex_claim + "\n\nProvide your assessment of this claim.", system=underwriter_system)
print(f"System: Underwriter")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

# ---- Persona 3: Policyholder Communications ----
print("\n\n--- Persona 3: Policyholder Communications ---")
comms_system = """You are a customer service specialist for an insurance company. 
Draft a clear, empathetic communication to the policyholder about their claim. 
Avoid jargon. Explain next steps in plain language. Keep it under 150 words."""

result = ask(complex_claim + "\n\nDraft a message to the policyholder about this claim.", system=comms_system)
print(f"System: Policyholder Communications")
print(f"Answer: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")


# ---- Persona impact on the SAME question ----
print(f"\n\n{'=' * 65}")
print("SAME QUESTION, DIFFERENT PERSONAS")
print(f"{'=' * 65}")

question = "Should we be concerned about fraud in this claim?"

personas = [
    ("Claims Adjuster", "You are a senior claims adjuster. Be direct and practical."),
    ("Fraud Investigator", "You are an insurance fraud investigator with the Special Investigations Unit. Analyze with professional skepticism. Flag specific indicators."),
    ("Defense Attorney", "You are an insurance defense attorney. Assess from a legal exposure perspective. Be precise about risk.")
]

for name, system in personas:
    print(f"\n--- {name} ---")
    result = ask(f"{complex_claim}\n\nQuestion: {question}", system=system)
    print(f"Answer: {result['text'][:300]}...")
    print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")

print(f"\n{'=' * 65}")
print("Same facts, same question — different personas produce different")
print("insights. Role-based prompting lets you get multiple perspectives")
print("from a single AI system, each tuned to a specific business need.")
print(f"{'=' * 65}")

In [ ]:
# =============================================================================
# CELL 4: Structured Output Extraction
# =============================================================================
# Technique: Prompt engineering for consistent, parseable JSON output
# Why it matters: Downstream systems need structured data, not free text
# =============================================================================

import json

def parse_llm_json(text):
    """Strip markdown fencing and parse JSON from LLM output."""
    raw = text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

# --- A complex claim scenario for extraction ---
complex_claim = """
CLAIM REPORT - Filed February 14, 2026

Policyholder: Maria Esperanza-Gutierrez, Policy #HO-2024-88712
Property: 45 Riverside Drive, Unit 12B, Hartford, CT 06106
Coverage: HO-3 Special Form, $485,000 dwelling, $242,500 personal property
Deductible: $2,500

Incident Date: February 10, 2026 at approximately 2:15 AM
Reported Date: February 11, 2026

Description of Loss:
A pipe burst in the upstairs bathroom during a cold snap (outside temp recorded 
at -14°F). Water flowed for approximately 3 hours before the policyholder was 
awakened by the sound. Water damage affected the second floor bathroom, hallway, 
and first floor living room ceiling. Hardwood floors in the hallway are buckling. 
The living room ceiling has a visible 4-foot sag and water staining. Personal 
property damaged includes a Persian rug (purchased 2019, $8,200), a leather sofa 
($3,400), and electronics equipment ($1,950).

Emergency Mitigation: ServPro arrived February 11 at 9 AM. Industrial fans and 
dehumidifiers placed. Temporary ceiling support installed.

Preliminary Damage Estimate:
- Plumbing repair: $1,800
- Bathroom restoration: $12,500  
- Hallway flooring: $8,900
- Ceiling repair and repaint: $6,200
- Personal property: $13,550
- Emergency mitigation: $3,400
Total Estimated: $46,350

Prior Claims: Wind damage claim (2022, $4,200 paid), Theft claim (2023, $2,100 paid)
"""

# --- Task 1: Zero-shot JSON extraction (baseline) ---
print("=" * 70)
print("TASK 1: Zero-Shot JSON Extraction")
print("=" * 70)

zero_shot_json = f"""Extract the following from this insurance claim into JSON format:
- claimant name
- policy number  
- incident date
- cause of loss
- total estimate
- line items with descriptions and amounts
- prior claims

CLAIM:
{complex_claim}"""

result = ask(zero_shot_json)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Task 2: Schema-driven extraction (tell it EXACTLY what you want) ---
print("\n" + "=" * 70)
print("TASK 2: Schema-Driven JSON Extraction")
print("=" * 70)

schema_prompt = f"""Extract structured data from this insurance claim. 
Return ONLY valid JSON matching this exact schema — no markdown, no explanation:

{{
  "claimant": {{
    "full_name": "string",
    "policy_number": "string",
    "property_address": "string"
  }},
  "coverage": {{
    "form_type": "string",
    "dwelling_limit": number,
    "personal_property_limit": number,
    "deductible": number
  }},
  "incident": {{
    "date": "YYYY-MM-DD",
    "time": "HH:MM",
    "cause_of_loss": "string (one of: fire, water, wind, theft, liability, other)",
    "description_summary": "string (max 50 words)"
  }},
  "damages": {{
    "line_items": [
      {{"category": "string", "description": "string", "amount": number}}
    ],
    "total_estimate": number,
    "above_deductible": boolean
  }},
  "risk_flags": {{
    "prior_claim_count": number,
    "reported_delay_days": number,
    "late_night_incident": boolean
  }}
}}

CLAIM:
{complex_claim}"""

result = ask(schema_prompt)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Validate that we got parseable JSON
try:
    parsed = parse_llm_json(result['text'])
    print("\n✓ Valid JSON — successfully parsed")
    print(f"  Claimant: {parsed['claimant']['full_name']}")
    print(f"  Cause: {parsed['incident']['cause_of_loss']}")
    print(f"  Total: ${parsed['damages']['total_estimate']:,.2f}")
    print(f"  Above deductible: {parsed['damages']['above_deductible']}")
    print(f"  Risk flags: {parsed['risk_flags']}")
except json.JSONDecodeError as e:
    print(f"\n✗ JSON parse error: {e}")

# --- Task 3: Few-shot + schema for edge cases ---
print("\n" + "=" * 70)
print("TASK 3: Few-Shot JSON — Handling Missing & Ambiguous Data")
print("=" * 70)

# Intentionally vague claim to test robustness
vague_claim = """
Got a call from Tom Baker about his car. Says someone hit it in a parking 
lot last week sometime, maybe Thursday? He's got full coverage on his 
2021 Civic. Wants to know if it's worth filing. Bumper is cracked and 
there's a dent in the quarter panel. He didn't get the other driver's info. 
No police report.
"""

fewshot_schema = f"""Extract structured claim data as JSON. When information is 
missing or ambiguous, use null for unknown values and add an "extraction_notes" 
array explaining what's missing.

EXAMPLE INPUT:
"Jane called about water in her basement. Happened over the weekend. She has 
homeowners insurance."

EXAMPLE OUTPUT:
{{
  "claimant": {{"name": "Jane", "policy_number": null}},
  "incident": {{
    "date": null,
    "cause": "water",
    "loss_type": "property",
    "description": "Water in basement, timing uncertain"
  }},
  "damages": {{"estimate": null, "items": []}},
  "extraction_notes": [
    "No last name provided",
    "No policy number — needs lookup",
    "Exact date unknown — 'over the weekend'",
    "No damage estimate yet",
    "Cause of water entry not specified — could be flood, pipe, or groundwater"
  ],
  "data_completeness": 0.3
}}

NOW EXTRACT FROM THIS:
{vague_claim}"""

result = ask(fewshot_schema)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Validate
try:
    parsed = parse_llm_json(result['text'])
    print(f"\n✓ Valid JSON — completeness score: {parsed.get('data_completeness', 'N/A')}")
    if 'extraction_notes' in parsed:
        print(f"  Missing data flags: {len(parsed['extraction_notes'])}")
        for note in parsed['extraction_notes']:
            print(f"    • {note}")
except json.JSONDecodeError as e:
    print(f"\n✗ JSON parse error: {e}")

In [ ]:
# =============================================================================
# CELL 5: Technique Comparison & Summary
# =============================================================================
# Compare all four prompting techniques on the SAME task to see differences
# in output quality, consistency, and token efficiency
# =============================================================================

# --- Same claim for all techniques ---
comparison_claim = """
Policyholder James Chen called to report that his 2023 Tesla Model Y was 
damaged in his garage. He states that on February 20, 2026, he was charging 
the vehicle overnight when he smelled smoke around 3 AM. The charging unit 
malfunctioned, causing scorch marks on the driver's side rear quarter panel 
and melting the charge port. The garage wall also has heat damage. Fire 
department responded but no open flames were found. Vehicle is drivable but 
charge port is non-functional. He has photos and the fire department report. 
Policy: AUTO-2025-33190, full coverage with $1,000 deductible.
"""

target_task = "Extract a structured claim summary suitable for an adjuster's queue."

results = {}

# --- Technique 1: Zero-Shot ---
print("=" * 70)
print("TECHNIQUE 1: Zero-Shot")
print("=" * 70)

zs_prompt = f"""{target_task}

CLAIM:
{comparison_claim}"""

r = ask(zs_prompt)
results['zero_shot'] = r
print(r['text'])
print(f"\n[Tokens: {r['input_tokens']} in / {r['output_tokens']} out]")

# --- Technique 2: Few-Shot ---
print("\n" + "=" * 70)
print("TECHNIQUE 2: Few-Shot")
print("=" * 70)

fs_prompt = f"""{target_task}

EXAMPLE:
Input: "Sandra Miller reports a tree fell on her roof during last night's storm. 
Several shingles damaged and a skylight is cracked. Policy HO-2024-55123, $2,500 
deductible."

Output:
CLAIM QUEUE ENTRY
Insured: Miller, Sandra | Policy: HO-2024-55123
Peril: Wind/Storm | Priority: MEDIUM
Loss: Tree impact — roof shingle damage, cracked skylight
Deductible: $2,500 | Est. Severity: $8,000-$15,000
Action: Assign field adjuster, request contractor estimate
Flags: None

NOW PROCESS THIS CLAIM:
{comparison_claim}"""

r = ask(fs_prompt)
results['few_shot'] = r
print(r['text'])
print(f"\n[Tokens: {r['input_tokens']} in / {r['output_tokens']} out]")

# --- Technique 3: Role-Based ---
print("\n" + "=" * 70)
print("TECHNIQUE 3: Role-Based")
print("=" * 70)

role_system = """You are a senior auto claims supervisor with 20 years of experience. 
You triage incoming claims for your team of adjusters. You prioritize based on 
complexity, liability exposure, and potential for subrogation. You flag anything 
unusual that junior adjusters might miss."""

r = ask(f"""{target_task}

CLAIM:
{comparison_claim}""", system=role_system)
results['role_based'] = r
print(r['text'])
print(f"\n[Tokens: {r['input_tokens']} in / {r['output_tokens']} out]")

# --- Technique 4: Structured Output (JSON) ---
print("\n" + "=" * 70)
print("TECHNIQUE 4: Structured Output (JSON)")
print("=" * 70)

json_prompt = f"""{target_task}
Return ONLY valid JSON, no markdown fencing, matching this schema:

{{
  "queue_entry": {{
    "insured": "string (Last, First)",
    "policy": "string",
    "peril": "string",
    "priority": "string (LOW | MEDIUM | HIGH | URGENT)",
    "loss_summary": "string (max 30 words)",
    "deductible": number,
    "estimated_severity": "string (dollar range)",
    "recommended_action": ["string"],
    "flags": ["string"]
  }}
}}

CLAIM:
{comparison_claim}"""

r = ask(json_prompt)
results['structured'] = r
print(r['text'])
print(f"\n[Tokens: {r['input_tokens']} in / {r['output_tokens']} out]")

# Validate JSON
try:
    parsed = parse_llm_json(r['text'])
    print("\n✓ Valid JSON")
except json.JSONDecodeError as e:
    print(f"\n✗ JSON parse error: {e}")

# --- Comparison Dashboard ---
print("\n" + "=" * 70)
print("TECHNIQUE COMPARISON DASHBOARD")
print("=" * 70)

print(f"\n{'Technique':<20} {'Input Tok':>10} {'Output Tok':>11} {'Total Tok':>10} {'Cost Est':>10}")
print("-" * 65)

# Claude Sonnet pricing: $3/M input, $15/M output
for name, r in results.items():
    total = r['input_tokens'] + r['output_tokens']
    cost = (r['input_tokens'] * 3 + r['output_tokens'] * 15) / 1_000_000
    label = name.replace('_', '-').title()
    print(f"{label:<20} {r['input_tokens']:>10,} {r['output_tokens']:>11,} {total:>10,} {f'${cost:.4f}':>10}")

print(f"\n{'─' * 65}")
print("KEY TAKEAWAYS:")
print("─" * 65)
print("""
1. ZERO-SHOT is fastest to write but output format varies between calls.
   → Best for: exploration, one-off analysis, prototyping

2. FEW-SHOT locks in a consistent format without a rigid schema.
   → Best for: human-readable reports, standardized but flexible output

3. ROLE-BASED adds domain reasoning the other techniques miss.
   → Best for: expert analysis, risk flagging, nuanced judgment calls

4. STRUCTURED (JSON) is machine-parseable and pipeline-ready.
   → Best for: downstream automation, API responses, database ingestion

In production, you typically COMBINE techniques:
   Role-based system prompt + Few-shot examples + JSON schema
   = Expert-quality, consistent, machine-readable output
""")